In [0]:

CREATE TABLE IF NOT EXISTS workspace.myra_invest.gold_stock_news_score (

    symbol STRING,
    companyName STRING,

    newsDate DATE,

    totalArticles INT,

    positiveArticles INT,
    neutralArticles INT,
    negativeArticles INT,

    averageSentimentScore DOUBLE,
    averageConfidenceScore DOUBLE,

    newsStrength DOUBLE,

    ingestionDate DATE

)
USING DELTA;

SHOW TABLES IN workspace.myra_invest;

In [0]:
%python

from pyspark.sql.functions import *

silver_df = spark.table("workspace.myra_invest.silver_stock_news")

print(f"Silver News Records: {silver_df.count()}")

display(silver_df.limit(5))

In [0]:
%python
gold_df = (
    silver_df
      .groupBy(
          "symbol",
          "companyName",
          "ingestionDate"
      )
      .agg(

          count("*").alias("totalArticles"),

          sum(
              when(col("sentiment") == "POSITIVE", 1).otherwise(0)
          ).alias("positiveArticles"),

          sum(
              when(col("sentiment") == "NEUTRAL", 1).otherwise(0)
          ).alias("neutralArticles"),

          sum(
              when(col("sentiment") == "NEGATIVE", 1).otherwise(0)
          ).alias("negativeArticles"),

          round(avg("sentimentScore"),2)
              .alias("averageSentimentScore"),

          round(avg("confidenceScore"),2)
              .alias("averageConfidenceScore")
      )
      .withColumnRenamed("ingestionDate","newsDate")
)

In [0]:
%python
gold_df = (
    gold_df
      .withColumn(
          "newsStrength",
          round(
              col("averageSentimentScore") *
              col("averageConfidenceScore") *
              log1p(col("totalArticles")),
              3
          )
      )
)

display(gold_df.orderBy(desc("newsStrength")))

In [0]:
%python
(
    gold_df.write
      .mode("overwrite")
      .option("overwriteSchema","true")
      .format("delta")
      .saveAsTable("workspace.myra_invest.gold_stock_news_score")
)

print("✅ Gold News Score table created successfully.")

gold_count = spark.table(
    "workspace.myra_invest.gold_stock_news_score"
).count()

print(f"Gold records: {gold_count}")

In [0]:
SELECT *
FROM workspace.myra_invest.gold_stock_news_score
ORDER BY newsStrength DESC;